# options_4h_jma_atr_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (32).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `options` |
| Trading style | `unknown` |
| Timeframe | `4h` |
| Code cells | 10 |
| Detected functions | resample_1hr_custom, compute_jma_and_signals, prepare_data, py_backtest_trades, calc_pnl, backtest_trades, compute_jma_fast, compute_normalized_jma_fast, compute_rs_volatility_fast, compute_rs_threshold_fast, generate_signals_fast, backtest_fast |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Binance_Crypto_T7_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2023-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['ETHUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def compute_jma_and_signals(df, smooth_length=14, normalization_period=72,
                                upper_threshold=1.75, lower_threshold=-0.75,
                                rs_period=55, rs_percentile=70):
    df = df.copy()
    price = df['close']

    # JMA calculation
    alpha = 2.0 / (smooth_length + 1)
    jma_values = [price.iloc[0]]
    for p in price.iloc[1:]:
        jma_values.append(jma_values[-1] + (p - jma_values[-1]) * alpha)
    df['JMA'] = jma_values

    # Normalization using shifted JMA to avoid look-ahead
    df['JMA_shifted'] = df['JMA'].shift(1)
    df['Normalized_JMA'] = (df['JMA_shifted'] - df['JMA_shifted'].rolling(normalization_period).mean()) / \
                          df['JMA_shifted'].rolling(normalization_period).std()

    # RS Volatility using previous bar's OHLC
    o = df['open'].shift(1).replace(0, np.nan)
    h = df['high'].shift(1)
    l = df['low'].shift(1)
    c = df['close'].shift(1).replace(0, np.nan)

    log_hc, log_ho = np.log(h / c), np.log(h / o)
    log_lc, log_lo = np.log(l / c), np.log(l / o)

    df['RS_Volatility'] = ((log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()) * 1e6
    df['RS_Threshold'] = df['RS_Volatility'].rolling(normalization_period).quantile(rs_percentile / 100.0)



    # Signals based on shifted/lagged data
    df['Signal'] = 0
    df.loc[(df['Normalized_JMA'] > upper_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 1
    df.loc[(df['Normalized_JMA'] < lower_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 2

    return df

def prepare_data(data):
    """
    Prepare data for backtesting by calculating indicators and signals
    Expected input: DataFrame with columns ['Date', 'Ticker', 'open', 'high', 'low', 'close']
    """
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    jma_results = [compute_jma_and_signals(group.reset_index(drop=True)) for ticker, group in data.groupby('Ticker')]
    data = pd.concat(jma_results, ignore_index=True)

    # Shift signals to avoid look-ahead bias
    data['Signal'] = data.groupby('Ticker', group_keys=False)['Signal'].shift(1)
    data = data.dropna(subset=['Signal'])
    data['Signal'] = data['Signal'].astype(np.int8)

    return data

def py_backtest_trades(data, ticker, capital, fee_rate, slippage_rate, max_trades, stop_loss_pct):
    def calc_pnl(entry_price, exit_price, qty, direction):
        pnl = (exit_price - entry_price) * qty * direction
        fee = fee_rate * (entry_price + exit_price) * qty
        return pnl - fee

    n = data.shape[0]
    trades, cumulative_pnl = [], 0.0
    trade_count = 0

    # Track multiple positions (up to max_trades)
    positions = []  # List of dictionaries containing position info

    for i in range(n):
        # Fixed indexing - ensure all values are properly extracted
        row = data[i]
        time = pd.to_datetime(row[0]) if not isinstance(row[0], pd.Timestamp) else row[0]
        open_ = float(row[1])
        high = float(row[2])
        low = float(row[3])
        close = float(row[4])
        signal = int(row[5])


        # Entry logic - take trade on every signal (1 or 2) if under max trades
        if signal in (1, 2) and trade_count < max_trades:
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate if direction == 1 else 1 - slippage_rate)
            position_capital = capital / max_trades  # Use 1/max_trades of capital per trade
            quantity = position_capital / entry_price
            sl_price = entry_price * ((1 - stop_loss_pct/100) if direction == 1 else (1 + stop_loss_pct/100))  # Convert percentage

            # Create new position
            position = {
                'entry_time': time,
                'entry_price': entry_price,
                'quantity': quantity,
                'direction': direction,
                'sl_price': sl_price  # Fixed SL that doesn't change
            }

            # Add position to active positions list
            positions.append(position)
            trade_count += 1

        # Exit logic - close positions on signal 0
        elif signal == 0 and positions:
            # Close all open positions
            positions_to_close = positions.copy()
            positions = []  # Clear positions list

            for pos in positions_to_close:
                # Exit at open price with slippage
                direction = pos['direction']
                exit_price = open_ * (1 - slippage_rate if direction == 1 else 1 + slippage_rate)

                net_pnl = calc_pnl(pos['entry_price'], exit_price, pos['quantity'], direction)
                cumulative_pnl += net_pnl

                # Consistent parameter order for all trades
                trades.append((ticker, pos['entry_time'], time, pos['entry_price'], exit_price,
                             pos['quantity'], net_pnl, "Signal Exit", cumulative_pnl,
                             "long" if direction == 1 else "short"))

            # Reset trade count since all positions are closed
            trade_count = 0

        # Check stop loss for open positions
        elif positions:
            positions_to_remove = []

            for j, pos in enumerate(positions):
                direction = pos['direction']

                # Check if fixed stop loss is hit
                if (direction == 1 and low <= pos['sl_price']) or (direction == -1 and high >= pos['sl_price']):
                    exit_price = pos['sl_price']
                    net_pnl = calc_pnl(pos['entry_price'], exit_price, pos['quantity'], direction)
                    cumulative_pnl += net_pnl

                    # Consistent parameter order
                    trades.append((ticker, pos['entry_time'], time, pos['entry_price'],
                                 exit_price, pos['quantity'], net_pnl, f"Fixed {stop_loss_pct}% SL", cumulative_pnl,
                                 "long" if direction == 1 else "short"))
                    positions_to_remove.append(j)

            # Remove closed positions (in reverse order to maintain indices)
            for j in reversed(positions_to_remove):
                positions.pop(j)

            # Update trade count after closing positions
            trade_count = len(positions)

    # Close any remaining open positions at the end
    for pos in positions:
        direction = pos['direction']
        # Use last available close price
        exit_price = close * (1 - slippage_rate if direction == 1 else 1 + slippage_rate)
        net_pnl = calc_pnl(pos['entry_price'], exit_price, pos['quantity'], direction)
        cumulative_pnl += net_pnl

        # Consistent parameter order
        trades.append((ticker, pos['entry_time'], time, pos['entry_price'], exit_price,
                     pos['quantity'], net_pnl, "End of Data", cumulative_pnl,
                     "long" if direction == 1 else "short"))

    return trades

def backtest_trades(data, initial_capital, fee_rate, slippage_rate, max_trades, stop_loss_pct):
    """
    Fixed function signature to include slippage_rate parameter
    """
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][["Date", "open", "high", "low", "close", "Signal"]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades = []
    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = py_backtest_trades(ticker_map[ticker], ticker, initial_capital, fee_rate, slippage_rate, max_trades, stop_loss_pct)
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "Qty",
                                           "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Usage - Fixed function call
data = prepare_data(data)  # Assuming 'data' is your input DataFrame
tradebook = backtest_trades(data, initial_capital=100000, fee_rate=0.00025, slippage_rate=0.0010, max_trades=4, stop_loss_pct=0.4)
tradebook.to_csv("/content/drive/MyDrive/tradebook_ETHUSDT.csv", index=False)
tradebook

In [ ]:
import pandas as pd
import numpy as np
from itertools import product
from tqdm import tqdm
import warnings
from numba import jit, types
from numba.typed import List
import pickle
import os
from datetime import datetime
import json
warnings.filterwarnings('ignore')

@jit(nopython=True)
def compute_jma_fast(prices, smooth_length):
    """Fast JMA calculation"""
    alpha = 2.0 / (smooth_length + 1)
    jma = np.empty(len(prices))
    jma[0] = prices[0]

    for i in range(1, len(prices)):
        jma[i] = jma[i-1] + (prices[i] - jma[i-1]) * alpha

    return jma

@jit(nopython=True)
def compute_normalized_jma_fast(jma, normalization_period):
    """Fast normalized JMA calculation"""
    n = len(jma)
    normalized = np.full(n, np.nan)

    for i in range(normalization_period, n):
        window = jma[max(0, i-normalization_period):i]
        mean_val = np.mean(window)
        std_val = np.std(window)

        if std_val > 0:
            normalized[i] = (jma[i-1] - mean_val) / std_val

    return normalized

@jit(nopython=True)
def compute_rs_volatility_fast(o, h, l, c, rs_period):
    """Fast RS volatility calculation"""
    n = len(o)
    rs_vol = np.full(n, np.nan)

    for i in range(1, n):
        o_prev = o[i-1]
        h_prev = h[i-1]
        l_prev = l[i-1]
        c_prev = c[i-1]

        if o_prev > 0 and c_prev > 0:
            log_hc = np.log(h_prev / c_prev)
            log_ho = np.log(h_prev / o_prev)
            log_lc = np.log(l_prev / c_prev)
            log_lo = np.log(l_prev / o_prev)

            rs_val = (log_hc * log_ho + log_lc * log_lo) * 1e6

            start_idx = max(0, i - rs_period + 1)
            if i >= rs_period:
                window_sum = 0.0
                count = 0
                for j in range(start_idx, i + 1):
                    if not np.isnan(rs_vol[j]) or j == i:
                        if j == i:
                            window_sum += rs_val
                        else:
                            window_sum += rs_vol[j]
                        count += 1
                rs_vol[i] = window_sum / count if count > 0 else rs_val
            else:
                rs_vol[i] = rs_val

    return rs_vol

@jit(nopython=True)
def compute_rs_threshold_fast(rs_vol, normalization_period, rs_percentile):
    """Fast RS threshold calculation"""
    n = len(rs_vol)
    threshold = np.full(n, np.nan)

    for i in range(normalization_period, n):
        window = rs_vol[max(0, i-normalization_period):i]
        valid_window = window[~np.isnan(window)]

        if len(valid_window) > 0:
            threshold[i] = np.percentile(valid_window, rs_percentile)

    return threshold

@jit(nopython=True)
def generate_signals_fast(norm_jma, rs_vol, rs_threshold, upper_thresh, lower_thresh):
    """Fast signal generation"""
    n = len(norm_jma)
    signals = np.zeros(n, dtype=np.int8)

    for i in range(n):
        if not np.isnan(norm_jma[i]) and not np.isnan(rs_vol[i]) and not np.isnan(rs_threshold[i]):
            if norm_jma[i] > upper_thresh and rs_vol[i] > rs_threshold[i]:
                signals[i] = 1
            elif norm_jma[i] < lower_thresh and rs_vol[i] > rs_threshold[i]:
                signals[i] = 2

    return signals

@jit(nopython=True)
def backtest_fast(prices_ohlc, signals, max_trades, stop_loss_pct, fee_rate=0.001, slippage_rate=0.001):
    """Fast backtesting with fixed Numba typing"""
    n = len(signals)
    trades_pnl = np.empty(0, dtype=np.float64)
    trades_exit_idx = np.empty(0, dtype=np.int64)

    # Use arrays instead of lists for positions
    # Each position: [entry_price, direction, sl_price, entry_idx]
    max_positions = max_trades
    positions = np.full((max_positions, 4), np.nan)
    position_count = 0
    trade_count = 0

    for i in range(n):
        open_price = prices_ohlc[i, 0]
        high_price = prices_ohlc[i, 1]
        low_price = prices_ohlc[i, 2]
        close_price = prices_ohlc[i, 3]
        signal = signals[i]

        # Entry logic
        if signal in (1, 2) and trade_count < max_trades:
            direction = 1.0 if signal == 1 else -1.0
            entry_price = open_price * (1 + slippage_rate if direction == 1 else 1 - slippage_rate)
            sl_price = entry_price * ((1 - stop_loss_pct/100) if direction == 1 else (1 + stop_loss_pct/100))

            # Add position to array
            if position_count < max_positions:
                positions[position_count] = [entry_price, direction, sl_price, float(i)]
                position_count += 1
                trade_count += 1

        # Exit on signal 0
        elif signal == 0 and position_count > 0:
            exit_price = open_price * (1 - slippage_rate if positions[0, 1] == 1 else 1 + slippage_rate)

            for pos_idx in range(position_count):
                entry_price = positions[pos_idx, 0]
                direction = positions[pos_idx, 1]
                sl_price = positions[pos_idx, 2]
                entry_idx = positions[pos_idx, 3]

                pnl = (exit_price - entry_price) * direction - fee_rate * (entry_price + exit_price)
                trades_pnl = np.append(trades_pnl, pnl)
                trades_exit_idx = np.append(trades_exit_idx, i)

            # Clear positions
            positions[:] = np.nan
            position_count = 0
            trade_count = 0

        # Check stop loss
        elif position_count > 0:
            new_position_count = 0
            temp_positions = np.full((max_positions, 4), np.nan)

            for pos_idx in range(position_count):
                entry_price = positions[pos_idx, 0]
                direction = positions[pos_idx, 1]
                sl_price = positions[pos_idx, 2]
                entry_idx = positions[pos_idx, 3]

                if (direction == 1 and low_price <= sl_price) or (direction == -1 and high_price >= sl_price):
                    pnl = (sl_price - entry_price) * direction - fee_rate * (entry_price + sl_price)
                    trades_pnl = np.append(trades_pnl, pnl)
                    trades_exit_idx = np.append(trades_exit_idx, i)
                else:
                    temp_positions[new_position_count] = positions[pos_idx]
                    new_position_count += 1

            positions = temp_positions
            position_count = new_position_count
            trade_count = position_count

    # Close remaining positions
    if position_count > 0:
        final_price = prices_ohlc[-1, 3]
        exit_price = final_price * (1 - slippage_rate if positions[0, 1] == 1 else 1 + slippage_rate)

        for pos_idx in range(position_count):
            entry_price = positions[pos_idx, 0]
            direction = positions[pos_idx, 1]
            sl_price = positions[pos_idx, 2]
            entry_idx = positions[pos_idx, 3]

            pnl = (exit_price - entry_price) * direction - fee_rate * (entry_price + exit_price)
            trades_pnl = np.append(trades_pnl, pnl)
            trades_exit_idx = np.append(trades_exit_idx, n-1)

    return trades_pnl, trades_exit_idx

@jit(nopython=True)
def calculate_metrics_fast(pnls, initial_capital=100000):
    """Fast metrics calculation"""
    if len(pnls) == 0:
        return 0.0, 0.0, 0.0, 0.0, 0, 0.0, 0.0, 0.0

    # Equity curve
    equity = initial_capital + np.cumsum(pnls)
    total_profit = equity[-1] - initial_capital

    # Drawdown
    peak = equity[0]
    max_dd = 0.0
    for i in range(len(equity)):
        if equity[i] > peak:
            peak = equity[i]
        dd = peak - equity[i]
        if dd > max_dd:
            max_dd = dd

    # Profit/DD ratio
    profit_dd_ratio = total_profit / max_dd if max_dd > 0 else (1000.0 if total_profit > 0 else 0.0)

    # Equity R²
    if len(equity) > 1:
        x = np.arange(len(equity))
        x_mean = np.mean(x)
        y_mean = np.mean(equity)

        num = np.sum((x - x_mean) * (equity - y_mean))
        den = np.sum((x - x_mean) ** 2)

        if den > 0:
            slope = num / den
            y_pred = slope * (x - x_mean) + y_mean
            ss_res = np.sum((equity - y_pred) ** 2)
            ss_tot = np.sum((equity - y_mean) ** 2)
            equity_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            equity_r2 = max(0.0, min(1.0, equity_r2))
        else:
            equity_r2 = 0.0
    else:
        equity_r2 = 0.0

    # Monthly consistency (simplified)
    positive_trades = np.sum(pnls > 0)
    monthly_consistency = positive_trades / len(pnls) if len(pnls) > 0 else 0.0

    # Win rate
    win_rate = positive_trades / len(pnls) if len(pnls) > 0 else 0.0

    # Composite score
    composite_score = (
        monthly_consistency * 0.4 +
        equity_r2 * 0.4 +
        min(max(profit_dd_ratio / 10, 0), 1) * 0.2
    )

    return equity_r2, monthly_consistency, profit_dd_ratio, win_rate, len(pnls), total_profit, max_dd, composite_score

def prepare_data_fast(data, smooth_length, normalization_period, upper_threshold, lower_threshold, rs_period, rs_percentile):
    """Fast data preparation"""
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'])

    results = []
    for ticker, group in data.groupby('Ticker'):
        group = group.sort_values('Date').reset_index(drop=True)

        prices = group['close'].values
        o = group['open'].values
        h = group['high'].values
        l = group['low'].values
        c = group['close'].values

        # Calculate indicators
        jma = compute_jma_fast(prices, smooth_length)
        norm_jma = compute_normalized_jma_fast(jma, normalization_period)
        rs_vol = compute_rs_volatility_fast(o, h, l, c, rs_period)
        rs_threshold = compute_rs_threshold_fast(rs_vol, normalization_period, rs_percentile)

        # Generate signals
        signals = generate_signals_fast(norm_jma, rs_vol, rs_threshold, upper_threshold, lower_threshold)

        # Shift signals
        signals = np.roll(signals, 1)
        signals[0] = 0

        group['Signal'] = signals
        results.append(group)

    return pd.concat(results, ignore_index=True)

def get_checkpoint_files(ticker_name, save_dir="/content/drive/MyDrive/"):
    """Get checkpoint file paths"""
    progress_file = os.path.join(save_dir, f"{ticker_name}_checkpoint_progress.json")
    results_file = os.path.join(save_dir, f"{ticker_name}_checkpoint_results.pkl")
    return progress_file, results_file

def save_checkpoint(ticker_name, completed_idx, total_combinations, results_list, save_dir="/content/drive/MyDrive/"):
    """Save checkpoint"""
    progress_file, results_file = get_checkpoint_files(ticker_name, save_dir)

    print(f"Attempting to save checkpoint to: {save_dir}")
    print(f"Progress file: {progress_file}")
    print(f"Results file: {results_file}")

    progress_data = {
        'completed_idx': completed_idx,
        'total_combinations': total_combinations,
        'timestamp': datetime.now().isoformat(),
        'ticker_name': ticker_name
    }

    try:
        # Check if directory exists and is writable
        if not os.path.exists(save_dir):
            print(f"Directory {save_dir} does not exist. Creating...")
            os.makedirs(save_dir, exist_ok=True)

        # Test write access
        test_file = os.path.join(save_dir, "test_write.tmp")
        with open(test_file, 'w') as f:
            f.write("test")
        os.remove(test_file)
        print("Directory is writable")

        # Save progress metadata
        with open(progress_file, 'w') as f:
            json.dump(progress_data, f, indent=2)
        print(f"Progress file saved: {os.path.exists(progress_file)}")

        # Save partial results
        with open(results_file, 'wb') as f:
            pickle.dump(results_list, f)
        print(f"Results file saved: {os.path.exists(results_file)}")
        print(f"Results file size: {os.path.getsize(results_file) if os.path.exists(results_file) else 0} bytes")

        print(f"✓ Checkpoint saved: {completed_idx+1}/{total_combinations} combinations completed")
        return True
    except Exception as e:
        print(f"Failed to save checkpoint: {e}")
        print(f"Error type: {type(e).__name__}")
        import traceback
        print(f"Full traceback: {traceback.format_exc()}")
        return False

def load_checkpoint(ticker_name, save_dir="/content/drive/MyDrive/"):
    """Load checkpoint"""
    progress_file, results_file = get_checkpoint_files(ticker_name, save_dir)

    if os.path.exists(progress_file) and os.path.exists(results_file):
        try:
            # Load progress
            with open(progress_file, 'r') as f:
                progress_data = json.load(f)

            # Load partial results
            with open(results_file, 'rb') as f:
                results_list = pickle.load(f)

            print(f"✓ Checkpoint loaded: Resuming from combination {progress_data['completed_idx']+1}")
            return progress_data['completed_idx'], results_list

        except Exception as e:
            print(f"Warning: Could not load checkpoint: {e}")
            return -1, []

    return -1, []

def cleanup_checkpoint(ticker_name, save_dir="/content/drive/MyDrive/"):
    """Clean up checkpoint files"""
    progress_file, results_file = get_checkpoint_files(ticker_name, save_dir)

    try:
        if os.path.exists(progress_file):
            os.remove(progress_file)
        if os.path.exists(results_file):
            os.remove(results_file)
        print("✓ Checkpoint files cleaned up")
    except Exception as e:
        print(f"Warning: Could not clean up checkpoint files: {e}")

def optimize_with_checkpoint(data, ticker_name, param_grid, resume=True, save_dir="/content/drive/MyDrive/"):
    """Enhanced optimization with checkpointing"""

    if ticker_name not in data['Ticker'].unique():
        print(f"Ticker '{ticker_name}' not found")
        return None

    ticker_data = data[data['Ticker'] == ticker_name].copy()
    if len(ticker_data) < 50:
        print(f"Insufficient data for {ticker_name}")
        return None

    os.makedirs(save_dir, exist_ok=True)

    # Generate parameter combinations
    keys, values = zip(*param_grid.items())
    combinations = list(product(*values))
    total_combinations = len(combinations)

    # Try loading checkpoint
    start_idx, results = (-1, [])
    if resume:
        start_idx, results = load_checkpoint(ticker_name, save_dir)

    if start_idx >= 0:
        print(f"✓ Resuming {ticker_name}: {start_idx+1}/{total_combinations} already completed")
        print(f"  Loaded {len(results)} results from checkpoint")
        remaining_combinations = combinations[start_idx + 1:]
    else:
        print(f"Starting fresh optimization for {ticker_name}")
        remaining_combinations = combinations

        # Only test checkpoint save on brand new run
        print("Testing checkpoint save capability...")
        if save_checkpoint(ticker_name, -1, total_combinations, [], save_dir):
            print("✓ Checkpoint test successful")
            cleanup_checkpoint(ticker_name, save_dir)
        else:
            print("⚠️  Checkpoint test failed (continuing without checkpoints)")

    # Progress bar
    pbar = tqdm(enumerate(remaining_combinations),
                total=len(remaining_combinations),
                desc=f"Optimizing {ticker_name}")

    for current_idx, combo in pbar:
        absolute_idx = (start_idx + 1) + current_idx
        params = dict(zip(keys, combo))

        try:
            max_trades = params.pop('max_trades')
            stop_loss_pct = params.pop('stop_loss_pct')

            # Prepare and backtest
            prepared = prepare_data_fast(ticker_data, **params)
            if prepared.empty:
                continue

            ohlc = prepared[['open', 'high', 'low', 'close']].values
            signals = prepared['Signal'].values
            pnls, exit_indices = backtest_fast(ohlc, signals, max_trades, stop_loss_pct)

            # Metrics
            equity_r2, monthly_consistency, profit_dd_ratio, win_rate, total_trades, total_profit, max_dd, composite_score = calculate_metrics_fast(pnls)

            # Save result
            result = params.copy()
            result.update({
                'max_trades': max_trades,
                'stop_loss_pct': stop_loss_pct,
                'Ticker': ticker_name,
                'Composite_Score': composite_score,
                'Equity_R2': equity_r2,
                'Monthly_Consistency': monthly_consistency,
                'Profit_DD_Ratio': profit_dd_ratio,
                'Win_Rate': win_rate,
                'Total_Trades': total_trades,
                'Total_Profit': total_profit,
                'Max_Drawdown': max_dd
            })
            results.append(result)

            # Show best score so far
            if results:
                best_score = max(r['Composite_Score'] for r in results)
                pbar.set_postfix({'Best Score': f'{best_score:.6f}', 'Results': len(results)})

        except Exception as e:
            print(f"⚠️ Error at combination {absolute_idx+1}: {e}")
            continue

        # Save checkpoint every 100 combos
        if (current_idx + 1) % 1000 == 0:
            print(f"\n💾 Saving checkpoint at {absolute_idx+1}/{total_combinations}...")
            if not save_checkpoint(ticker_name, absolute_idx, total_combinations, results, save_dir):
                print("⚠️ WARNING: Failed to save checkpoint!")

    pbar.close()

    if not results:
        print("No valid results found")
        return None

    # Final results save
    results_df = pd.DataFrame(results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = os.path.join(save_dir, f"{ticker_name}_optimization_{timestamp}.csv")
    pkl_path = os.path.join(save_dir, f"{ticker_name}_optimization_{timestamp}.pkl")

    results_df.to_csv(csv_path, index=False)
    with open(pkl_path, 'wb') as f:
        pickle.dump({
            'results_df': results_df,
            'ticker': ticker_name,
            'date': datetime.now(),
            'param_grid': param_grid,
            'total_combinations': total_combinations
        }, f)

    print(f"\n✓ Final results saved:\n  CSV: {csv_path}\n  PKL: {pkl_path}")

    # Clean up after success
    cleanup_checkpoint(ticker_name, save_dir)

    # Summary
    print(f"\n✓ Optimization completed for {ticker_name}")
    print(f"  Total combinations tested: {total_combinations:,}")
    print(f"  Valid results: {len(results_df):,}")
    print(f"  Best composite score: {results_df['Composite_Score'].max():.6f}")

    top_results = results_df.nlargest(5, 'Composite_Score')
    print("\nTop 5 parameter combinations:")
    for i, (_, row) in enumerate(top_results.iterrows(), 1):
        print(f"  {i}. Score: {row['Composite_Score']:.6f} | "
              f"R²: {row['Equity_R2']:.3f} | "
              f"Consistency: {row['Monthly_Consistency']:.3f} | "
              f"P/DD: {row['Profit_DD_Ratio']:.2f}")

    return results_df

def get_param_grid():
    """Parameter grid"""
    return {
        'smooth_length': [14],
        'normalization_period': [36, 48, 60, 72, 84, 96],
        'upper_threshold': [1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5],
        'lower_threshold': [-0.75, -1.0, -1.25, -1.5, -1.75, -2.0, -2.25],
        'rs_period': [3, 7, 14, 21, 34, 55],
        'rs_percentile': [10, 30, 50, 70],
        'max_trades': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'stop_loss_pct': [0.3, 0.4, 0.5, 0.75, 1.0, 1.5]
    }

def run_optimization(data, ticker_name, param_grid=None, resume=True, save_dir="/content/drive/MyDrive/"):
    """Run optimization with checkpointing support"""

    if param_grid is None:
        param_grid = get_param_grid()

    return optimize_with_checkpoint(data, ticker_name, param_grid, resume=resume, save_dir=save_dir)

def list_checkpoints(save_dir="/content/drive/MyDrive/"):
    """List available checkpoints"""
    if not os.path.exists(save_dir):
        print("No save directory found")
        return

    checkpoint_files = [f for f in os.listdir(save_dir)
                       if f.endswith("_checkpoint_progress.json")]

    if not checkpoint_files:
        print("No checkpoints found")
        return

    print("Available checkpoints:")
    for progress_file in sorted(checkpoint_files):
        progress_path = os.path.join(save_dir, progress_file)
        ticker_name = progress_file.replace("_checkpoint_progress.json", "")

        try:
            with open(progress_path, 'r') as f:
                progress_data = json.load(f)
            print(f"  {ticker_name}: {progress_data['completed_idx']+1}/{progress_data.get('total_combinations', '?')} completed")
        except:
            print(f"  {ticker_name}: (corrupted)")

def delete_checkpoint(ticker_name, save_dir="/content/drive/MyDrive/"):
    """Delete checkpoint for a specific ticker"""
    cleanup_checkpoint(ticker_name, save_dir)
    print(f"Checkpoint for {ticker_name} deleted")

# Usage
if __name__ == "__main__":
    print("Enhanced JMA Optimization System with Fixed Numba Typing")
    print("Usage:")
    print("  # Start new optimization or resume existing")
    print("  results = run_optimization(data, 'ETHUSDT')")
    print("  ")
    print("  # Force start from beginning (ignore checkpoints)")
    print("  results = run_optimization(data, 'ETHUSDT', resume=False)")
    print("  ")
    print("  # List available checkpoints")
    print("  list_checkpoints()")
    print("  ")
    print("  # Delete checkpoint for specific ticker")
    print("  delete_checkpoint('ETHUSDT')")

    grid = get_param_grid()
    total = np.prod([len(v) for v in grid.values()])
    print(f"\nTotal combinations per ticker: {total:,}")
    print(f"Checkpoints saved every: 100 combinations")
    print(f"Files saved directly to: /content/drive/MyDrive/")

In [ ]:
results = run_optimization(data, 'ETHUSDT')